# H1 Feature Engineering: Opponent Strength

This notebook prepares game-level data to compare Team A and Team C after accounting for opponent strength.

| Data source | Use in H1 | Reason |
|---|---|---|
| `regular_season_only.csv.gz` | Game outcomes, teams, dates, and locations | H1 concerns regular-season performance, so conference tournament games are excluded. |
| `team_strength_preseason.csv` | Preseason ratings for each team | Ratings are available before the season’s games and provide a measure of team strength without using those game outcomes. |

We use seasons 2015–2019 and 2021–2026. We omit 2020 from both hypotheses because tournament outcome data is unavailable for H2 in that season, allowing the analyses to use a consistent set of seasons.

The 2019 location values are marked unreliable. We retain those games but record their location as `Unknown`, so unreliable values do not affect the location comparison.

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = (
    Path.home()
    / "Documents"
    / "GitHub"
    / "CSC-4913-Project-4"
    / "cleaned_data"
)

print("Looking for data in:", DATA_DIR)
print("Folder found:", DATA_DIR.exists())

def read_cleaned_csv(filename):
    path = DATA_DIR / filename

    # Windows may hide the final .gz extension in File Explorer.
    if not path.exists() and filename.endswith(".csv.gz"):
        path = DATA_DIR / filename[:-3]

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename} in {DATA_DIR}. "
            f"Files in that folder: {[p.name for p in DATA_DIR.iterdir()]}"
        )

    return pd.read_csv(path)

Looking for data in: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\cleaned_data
Folder found: True


### Observation Unit and Prepared Variables

The observation unit is one team’s perspective in one regular-season game. Each game produces two rows: one for the winning team and one for the losing team.

| Column | Meaning | Role |
|---|---|---|
| `season` | Season in which the game was played | Defines the time-based split |
| `focal_team_id` | Team represented by this row | Identifies the team whose outcome is recorded |
| `opponent_team_id` | The other team in the game | Identifies the opponent |
| `focal_location` | Home, away, neutral, or `Unknown` for the focal team | Predictor |
| `focal_rating` | Focal team’s preseason rating | Used to represent the focal team’s strength |
| `opponent_rating` | Opponent’s preseason rating | Added in the opponent-adjusted model |
| `outcome` | `win` or `loss` for the focal team | Outcome to predict |

For example, a home win by Team A over Team X produces a Team A row with `focal_location = H` and `outcome = win`, and a Team X row with `focal_location = A` and `outcome = loss`.

In [9]:
games = regular_games.reset_index(drop=True).copy()
games["game_id"] = [f"G{i + 1:06d}" for i in range(len(games))]

# Keep location only when the data marks it reliable.
valid_location = games["w_loc"].isin(["H", "A", "N"])
location_is_reliable = (
    games["location_reliable"].fillna(False).eq(True)
    & valid_location
)

winner_location = games["w_loc"].where(location_is_reliable, "Unknown")

# Reverse home and away for the losing team's perspective.
loser_location = winner_location.map({
    "H": "A",
    "A": "H",
    "N": "N",
    "Unknown": "Unknown",
}).fillna("Unknown")

winner_rows = pd.DataFrame({
    "game_id": games["game_id"],
    "season": games["season"],
    "day_num": games["day_num"],
    "focal_team_id": games["w_team_id"],
    "opponent_team_id": games["l_team_id"],
    "focal_location": winner_location,
    "outcome": "win",
})

loser_rows = pd.DataFrame({
    "game_id": games["game_id"],
    "season": games["season"],
    "day_num": games["day_num"],
    "focal_team_id": games["l_team_id"],
    "opponent_team_id": games["w_team_id"],
    "focal_location": loser_location,
    "outcome": "loss",
})

h1_games = pd.concat([winner_rows, loser_rows], ignore_index=True)

focal_ratings = ratings[
    ["season", "team_id", "preseason_rating"]
].rename(columns={
    "team_id": "focal_team_id",
    "preseason_rating": "focal_rating",
})

opponent_ratings = ratings[
    ["season", "team_id", "preseason_rating"]
].rename(columns={
    "team_id": "opponent_team_id",
    "preseason_rating": "opponent_rating",
})

h1_games = h1_games.merge(
    focal_ratings,
    on=["season", "focal_team_id"],
    how="left",
    validate="many_to_one",
)

h1_games = h1_games.merge(
    opponent_ratings,
    on=["season", "opponent_team_id"],
    how="left",
    validate="many_to_one",
)

missing_ratings = h1_games[
    ["focal_rating", "opponent_rating"]
].isna().sum()

if missing_ratings.any():
    raise ValueError(f"Some preseason ratings did not match:\n{missing_ratings}")

# Display both perspective rows for four sample games.
sample_game_ids = h1_games["game_id"].drop_duplicates().head(4)

sample = (
    h1_games[h1_games["game_id"].isin(sample_game_ids)]
    .sort_values("game_id")
    .reset_index(drop=True)
)

display(sample)

,game_id,season,day_num,focal_team_id,opponent_team_id,focal_location,outcome,focal_rating,opponent_rating
0,G000001,2015,11,1335,1107,A,win,-17.7,-6.3
1,G000001,2015,11,1107,1335,H,loss,-6.3,-17.7
2,G000002,2015,11,1367,1128,A,win,-8.2,-13.9
3,G000002,2015,11,1128,1367,H,loss,-13.9,-8.2
4,G000003,2015,11,1229,1130,A,win,1.7,-10.3
5,G000003,2015,11,1130,1229,H,loss,-10.3,1.7
6,G000004,2015,11,1141,1224,H,win,3.2,1.2
7,G000004,2015,11,1224,1141,A,loss,1.2,3.2


### Rating Groups and Model Comparisons

The Bayesian network uses categorical variables. We therefore group each preseason rating as `weaker`, `middle`, or `stronger`, using the one-third and two-thirds quantiles of the training teams’ ratings.

The cutoffs are calculated separately for each fold using training seasons only, then applied to that fold’s held-out season. This prevents information from the held-out season from determining its own rating groups.

| Model version | Predictors |
|---|---|
| Baseline | Focal team’s rating group and game location |
| Opponent-adjusted | Focal team’s rating group, opponent’s rating group, and game location |

The outcome for both versions is the focal team’s win or loss.

In [10]:
# Calculate the cutoffs once, using only the initial training seasons.
initial_training_seasons = list(range(2015, 2020))

initial_training_ratings = (
    ratings.loc[
        ratings["season"].isin(initial_training_seasons),
        ["season", "team_id", "preseason_rating"],
    ]
    .drop_duplicates(["season", "team_id"])
)

low_medium_cutoff, medium_high_cutoff = (
    initial_training_ratings["preseason_rating"]
    .quantile([1 / 3, 2 / 3])
)

if low_medium_cutoff >= medium_high_cutoff:
    raise ValueError("The training data did not produce two distinct cutoffs.")

LOW_MEDIUM_CUTOFF = float(low_medium_cutoff)
MEDIUM_HIGH_CUTOFF = float(medium_high_cutoff)

print(f"Low/medium cutoff: {LOW_MEDIUM_CUTOFF}")
print(f"Medium/high cutoff: {MEDIUM_HIGH_CUTOFF}")


def add_rating_groups(data):
    data = data.copy()
    bins = [
        float("-inf"),
        LOW_MEDIUM_CUTOFF,
        MEDIUM_HIGH_CUTOFF,
        float("inf"),
    ]
    labels = ["low", "medium", "high"]

    data["focal_rating_group"] = pd.cut(
        data["focal_rating"],
        bins=bins,
        labels=labels,
        include_lowest=True,
    ).astype(str)

    data["opponent_rating_group"] = pd.cut(
        data["opponent_rating"],
        bins=bins,
        labels=labels,
        include_lowest=True,
    ).astype(str)

    return data


# These are the predictors the model runs should use.
target_column = "outcome"
baseline_features = ["focal_rating_group", "focal_location"]
opponent_adjusted_features = [
    "focal_rating_group",
    "focal_location",
    "opponent_rating_group",
]

print("Baseline predictors:", baseline_features)
print("Opponent-adjusted predictors:", opponent_adjusted_features)

Low/medium cutoff: -4.4
Medium/high cutoff: 4.9
Baseline predictors: ['focal_rating_group', 'focal_location']
Opponent-adjusted predictors: ['focal_rating_group', 'focal_location', 'opponent_rating_group']


## Walk-Forward Folds

Each validation fold holds out one season. Training data expands by adding each completed validation season before the next fold.

| Fold | Training seasons | Held-out season |
|---|---|---:|
| Validation 1 | 2015–2019 | 2021 |
| Validation 2 | 2015–2019, 2021 | 2022 |
| Validation 3 | 2015–2019, 2021–2022 | 2023 |
| Validation 4 | 2015–2019, 2021–2023 | 2024 |
| Validation 5 | 2015–2019, 2021–2024 | 2025 |
| Final test | 2015–2019, 2021–2025 | 2026 |

Both model versions use the same folds. Games from a held-out season are kept out of training for that fold.

In [11]:
fold_specs = []
training_seasons = list(range(2015, 2020))

# Validate each later season after training on earlier seasons.
for validation_season in range(2021, 2026):
    fold_specs.append({
        "fold": f"validation_{validation_season}",
        "training_seasons": training_seasons.copy(),
        "heldout_season": validation_season,
        "heldout_role": "validation",
    })
    training_seasons.append(validation_season)

# Use all seasons through 2025 for the final 2026 test.
final_training_seasons = list(range(2015, 2020)) + list(range(2021, 2026))

fold_specs.append({
    "fold": "final_test_2026",
    "training_seasons": final_training_seasons,
    "heldout_season": 2026,
    "heldout_role": "test",
})

fold_tables = []
manifest_rows = []

for spec in fold_specs:
    included_fold_seasons = (
        spec["training_seasons"] + [spec["heldout_season"]]
    )

    fold_data = h1_games[
        h1_games["season"].isin(included_fold_seasons)
    ].copy()

    fold_data["fold"] = spec["fold"]
    fold_data["data_role"] = "train"
    fold_data.loc[
        fold_data["season"].eq(spec["heldout_season"]),
        "data_role",
    ] = spec["heldout_role"]

    # Reuse the same 2015–2019 cutoffs in every fold.
    fold_data = add_rating_groups(fold_data)
    fold_tables.append(fold_data)

    manifest_rows.append({
        "fold": spec["fold"],
        "training_seasons": ",".join(map(str, spec["training_seasons"])),
        "heldout_season": spec["heldout_season"],
        "heldout_role": spec["heldout_role"],
        "low_medium_cutoff": LOW_MEDIUM_CUTOFF,
        "medium_high_cutoff": MEDIUM_HIGH_CUTOFF,
    })

h1_fold_data = pd.concat(fold_tables, ignore_index=True)
h1_fold_manifest = pd.DataFrame(manifest_rows)

# Save under the main project folder, beside cleaned_data.
output_dir = PROJECT_DIR / "prepared_data" / "H1"
output_dir.mkdir(parents=True, exist_ok=True)

h1_games.to_csv(output_dir / "h1_game_level.csv", index=False)
h1_fold_data.to_csv(output_dir / "h1_fold_data.csv", index=False)
h1_fold_manifest.to_csv(output_dir / "h1_fold_manifest.csv", index=False)

print("Rating cutoffs and folds:")
display(h1_fold_manifest)

# Show paired rows from the actual final test season, not its training rows.
final_test_rows = h1_fold_data[
    h1_fold_data["fold"].eq("final_test_2026")
    & h1_fold_data["data_role"].eq("test")
]

final_test_game_ids = final_test_rows["game_id"].drop_duplicates().head(4)

display(
    final_test_rows[
        final_test_rows["game_id"].isin(final_test_game_ids)
    ]
    .sort_values("game_id")
    .reset_index(drop=True)
)

print("Saved prepared tables to:", output_dir)

Rating cutoffs and folds:


,fold,training_seasons,heldout_season,heldout_role,low_medium_cutoff,medium_high_cutoff
0,validation_2021,"2015,2016,2017,2018,2019",2021,validation,-4.4,4.9
1,validation_2022,"2015,2016,2017,2018,2019,2021",2022,validation,-4.4,4.9
2,validation_2023,"2015,2016,2017,2018,2019,2021,2022",2023,validation,-4.4,4.9
3,validation_2024,"2015,2016,2017,2018,2019,2021,2022,2023",2024,validation,-4.4,4.9
4,validation_2025,"2015,2016,2017,2018,2019,2021,2022,2023,2024",2025,validation,-4.4,4.9
5,final_test_2026,"2015,2016,2017,2018,2019,2021,2022,2023,2024,2025",2026,test,-4.4,4.9


,game_id,season,day_num,focal_team_id,opponent_team_id,focal_location,outcome,focal_rating,opponent_rating,fold,data_role,focal_rating_group,opponent_rating_group
0,G000529,2026,69,1261,1455,A,win,10.4,0.7,final_test_2026,test,high,medium
1,G000529,2026,69,1455,1261,H,loss,0.7,10.4,final_test_2026,test,medium,high
2,G000722,2026,120,1306,1399,A,win,-9.6,-11.9,final_test_2026,test,low,low
3,G000722,2026,120,1399,1306,H,loss,-11.9,-9.6,final_test_2026,test,low,low
4,G000850,2026,43,1299,1414,A,win,-10.1,-2.3,final_test_2026,test,low,medium
5,G000850,2026,43,1414,1299,H,loss,-2.3,-10.1,final_test_2026,test,medium,low
6,G001024,2026,107,1324,1369,A,win,14.3,1.0,final_test_2026,test,high,medium
7,G001024,2026,107,1369,1324,H,loss,1.0,14.3,final_test_2026,test,medium,high


Saved prepared tables to: C:\Users\lsadi\Documents\GitHub\CSC-4913-Project-4\prepared_data\H1
